Monte Carlo test of the Poisson models on 2024-2026 validation matches.

Each match is simulated 10,000 times by sampling home and away goals from the fitted Poisson means. The predicted result is the most common simulated outcome (home win, draw, or away win). Accuracy is the share of validation matches where that prediction matches the actual result.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

N_SIMS = 10_000
RNG = np.random.default_rng(42)
OUTCOMES = np.array(["home", "draw", "away"])

home_model = sm.load("../models/poisson_home.pickle")
away_model = sm.load("../models/poisson_away.pickle")
validation = pd.read_csv("../data/training/validation.csv", parse_dates=["date"])

validation["home_lambda"] = home_model.predict(validation)
validation["away_lambda"] = away_model.predict(validation)

print(f"Validation matches: {len(validation)}  {validation['date'].min().date()} -> {validation['date'].max().date()}")
print(f"Simulations per match: {N_SIMS:,}")
validation[["date", "home_team", "away_team", "home_score", "away_score", "home_lambda", "away_lambda"]].head()

Validation matches: 1839  2024-01-12 -> 2026-03-31
Simulations per match: 10,000
Out[1]: 
        date    home_team      away_team  ...  away_score  home_lambda  away_lambda
0 2024-01-12        Qatar        Lebanon  ...         0.0     2.525599     0.518129
1 2024-01-13     China PR     Tajikistan  ...         0.0     1.521400     1.106061
2 2024-01-13    Australia          India  ...         0.0     3.523909     0.429016
3 2024-01-13   Uzbekistan          Syria  ...         0.0     2.121557     0.735798
4 2024-01-13  Ivory Coast  Guinea-Bissau  ...         0.0     2.165617     0.470278

[5 rows x 7 columns]


,date,home_team,away_team,home_score,away_score,home_lambda,away_lambda
0,2024-01-12,Qatar,Lebanon,3.0,0.0,2.525599,0.518129
1,2024-01-13,China PR,Tajikistan,0.0,0.0,1.521400,1.106061
2,2024-01-13,Australia,India,2.0,0.0,3.523909,0.429016
3,2024-01-13,Uzbekistan,Syria,0.0,0.0,2.121557,0.735798
4,2024-01-13,Ivory Coast,Guinea-Bissau,2.0,0.0,2.165617,0.470278


For match i, draw home_goals ~ Poisson(home_lambda_i) and away_goals ~ Poisson(away_lambda_i), independently, N_SIMS times.

The three outcome probabilities are the share of simulations that finish home win, draw, or away win. The predicted scoreline is the most common simulated score.

In [2]:
home_lambda = validation["home_lambda"].to_numpy()
away_lambda = validation["away_lambda"].to_numpy()
n_matches = len(validation)

home_sim = RNG.poisson(home_lambda, size=(N_SIMS, n_matches))
away_sim = RNG.poisson(away_lambda, size=(N_SIMS, n_matches))

p_home = (home_sim > away_sim).mean(axis=0)
p_draw = (home_sim == away_sim).mean(axis=0)
p_away = (home_sim < away_sim).mean(axis=0)

pred_code = np.argmax(np.vstack([p_home, p_draw, p_away]), axis=0)
actual_code = np.where(
    validation["home_score"] > validation["away_score"],
    0,
    np.where(validation["home_score"] == validation["away_score"], 1, 2),
)

packed = home_sim * 100 + away_sim
score_mode = np.empty(n_matches, dtype=int)
for i in range(n_matches):
    values, counts = np.unique(packed[:, i], return_counts=True)
    score_mode[i] = values[counts.argmax()]

pred_home_goals = score_mode // 100
pred_away_goals = score_mode % 100

validation = validation.copy()
validation["p_home"] = p_home
validation["p_draw"] = p_draw
validation["p_away"] = p_away
validation["predicted_outcome"] = OUTCOMES[pred_code]
validation["actual_outcome"] = OUTCOMES[actual_code]
validation["correct_outcome"] = pred_code == actual_code
validation["predicted_score"] = (
    pred_home_goals.astype(str) + "-" + pred_away_goals.astype(str)
)
validation["actual_score"] = (
    validation["home_score"].astype(int).astype(str)
    + "-"
    + validation["away_score"].astype(int).astype(str)
)
validation["correct_score"] = (pred_home_goals == validation["home_score"].to_numpy()) & (
    pred_away_goals == validation["away_score"].to_numpy()
)
validation["predicted_prob"] = np.vstack([p_home, p_draw, p_away])[pred_code, np.arange(n_matches)]

print("Simulation complete.")
print(f"Mean P(home / draw / away): {p_home.mean():.3f} / {p_draw.mean():.3f} / {p_away.mean():.3f}")

Simulation complete.
Mean P(home / draw / away): 0.491 / 0.202 / 0.307


Outcome accuracy: most common simulated result vs actual home / draw / away.

Exact-score accuracy: most common simulated scoreline vs the real score.

Baselines: always pick home, and pick the higher pre-match Elo side (never a draw).

In [3]:
outcome_acc = validation["correct_outcome"].mean()
score_acc = validation["correct_score"].mean()
always_home_acc = (validation["actual_outcome"] == "home").mean()
elo_pred = np.where(validation["elo_diff"] > 0, "home", "away")
elo_acc = (elo_pred == validation["actual_outcome"]).mean()

one_hot = np.eye(3)[actual_code]
probs = np.column_stack([p_home, p_draw, p_away])
brier = ((probs - one_hot) ** 2).sum(axis=1).mean()

print("Accuracy on validation")
print(f"{'Monte Carlo 1X2':<28}{outcome_acc:>8.3%}")
print(f"{'Monte Carlo exact score':<28}{score_acc:>8.3%}")
print(f"{'Always home':<28}{always_home_acc:>8.3%}")
print(f"{'Higher Elo (no draws)':<28}{elo_acc:>8.3%}")
print(f"\nBrier score (lower is better): {brier:.3f}")
print(f"Predicted draws: {(validation['predicted_outcome'] == 'draw').sum()}")
print(f"Actual draws:    {(validation['actual_outcome'] == 'draw').sum()}")

print("\nConfusion (rows = actual, columns = predicted)")
confusion = pd.crosstab(
    validation["actual_outcome"],
    validation["predicted_outcome"],
    rownames=["actual"],
    colnames=["predicted"],
).reindex(index=OUTCOMES, columns=OUTCOMES, fill_value=0)
print(confusion.to_string())

print("\nAccuracy by actual result")
print(validation.groupby("actual_outcome")["correct_outcome"].agg(["mean", "count"]).reindex(OUTCOMES))

print("\nAccuracy by year")
print(validation.groupby(validation["date"].dt.year)["correct_outcome"].agg(["mean", "count"]))

Accuracy on validation
Monte Carlo 1X2              60.740%
Monte Carlo exact score      12.724%
Always home                  46.547%
Higher Elo (no draws)        60.196%

Brier score (lower is better): 0.506
Predicted draws: 0
Actual draws:    429

Confusion (rows = actual, columns = predicted)
predicted  home  draw  away
actual                     
home        742     0   114
draw        274     0   155
away        179     0   375

Accuracy by actual result
                    mean  count
actual_outcome                 
home            0.866822    856
draw            0.000000    429
away            0.676895    554

Accuracy by year
          mean  count
date                 
2024  0.584008    988
2025  0.644357    762
2026  0.550562     89


Per-match predictions. correct_outcome is True when the simulated 1X2 pick matches the real result.

In [4]:
match_cols = [
    "date",
    "home_team",
    "away_team",
    "actual_score",
    "predicted_score",
    "actual_outcome",
    "predicted_outcome",
    "p_home",
    "p_draw",
    "p_away",
    "correct_outcome",
    "correct_score",
]

match_results = validation[match_cols].copy()
out_path = Path("../data/training/validation_predictions.csv")
match_results.to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(match_results)} matches)")
print(f"Correct outcomes: {validation['correct_outcome'].sum()} / {len(validation)}")
print(f"Correct scores:   {validation['correct_score'].sum()} / {len(validation)}")
match_results.head(20)

Saved ../data/training/validation_predictions.csv  (1839 matches)
Correct outcomes: 1117 / 1839
Correct scores:   234 / 1839
Out[1]: 
         date             home_team  ... correct_outcome correct_score
0  2024-01-12                 Qatar  ...            True         False
1  2024-01-13              China PR  ...           False         False
2  2024-01-13             Australia  ...            True         False
3  2024-01-13            Uzbekistan  ...           False         False
4  2024-01-13           Ivory Coast  ...            True          True
5  2024-01-14                 Ghana  ...           False         False
6  2024-01-14                 Egypt  ...           False         False
7  2024-01-14               Nigeria  ...           False         False
8  2024-01-14                 Japan  ...            True         False
9  2024-01-14                  Iran  ...            True         False
10 2024-01-14  United Arab Emirates  ...            True         False
11 2024-01-15 

,date,home_team,away_team,actual_score,predicted_score,actual_outcome,predicted_outcome,p_home,p_draw,p_away,correct_outcome,correct_score
0,2024-01-12,Qatar,Lebanon,3-0,2-0,home,home,0.8182,0.1321,0.0497,True,False
1,2024-01-13,China PR,Tajikistan,0-0,1-1,draw,home,0.4614,0.2564,0.2822,False,False
2,2024-01-13,Australia,India,2-0,3-0,home,home,0.9188,0.0600,0.0212,True,False
3,2024-01-13,Uzbekistan,Syria,0-0,2-0,draw,home,0.6866,0.1960,0.1174,False,False
4,2024-01-13,Ivory Coast,Guinea-Bissau,2-0,2-0,home,home,0.7768,0.1579,0.0653,True,True
5,2024-01-14,Ghana,Cape Verde,1-2,1-0,away,home,0.4916,0.2738,0.2346,False,False
6,2024-01-14,Egypt,Mozambique,2-2,2-0,draw,home,0.7426,0.1713,0.0861,False,False
7,2024-01-14,Nigeria,Equatorial Guinea,1-1,1-0,draw,home,0.5650,0.2371,0.1979,False,False
8,2024-01-14,Japan,Vietnam,4-2,3-0,home,home,0.9102,0.0687,0.0211,True,False
9,2024-01-14,Iran,Palestine,4-1,2-0,home,home,0.8297,0.1213,0.0490,True,False


Highest-confidence misses: matches where the model was sure and still wrong.

In [5]:
misses = (
    validation.loc[~validation["correct_outcome"], match_cols + ["predicted_prob"]]
    .sort_values("predicted_prob", ascending=False)
    .head(10)
)
misses

Out[1]: 
           date                 home_team  ... correct_score predicted_prob
949  2024-12-12                 Indonesia  ...         False         0.9823
1295 2025-08-14          Marshall Islands  ...         False         0.9743
1042 2025-03-21                    Guinea  ...         False         0.9595
171  2024-03-26                    Brunei  ...         False         0.9549
1681 2025-11-26                      Oman  ...         False         0.9482
118  2024-03-21              South Africa  ...         False         0.9468
1772 2026-03-26                  Tanzania  ...         False         0.9304
1799 2026-03-27               New Zealand  ...         False         0.9235
1296 2025-08-16  Turks and Caicos Islands  ...         False         0.9181
235  2024-06-06               Afghanistan  ...         False         0.9160

[10 rows x 13 columns]


,date,home_team,away_team,actual_score,predicted_score,actual_outcome,predicted_outcome,p_home,p_draw,p_away,correct_outcome,correct_score,predicted_prob
949,2024-12-12,Indonesia,Laos,3-3,5-0,draw,home,0.9823,0.0138,0.0039,False,False,0.9823
1295,2025-08-14,Marshall Islands,United States Virgin Islands,0-4,4-0,away,home,0.9743,0.0205,0.0052,False,False,0.9743
1042,2025-03-21,Guinea,Somalia,0-0,4-0,draw,home,0.9595,0.0309,0.0096,False,False,0.9595
171,2024-03-26,Brunei,Vanuatu,3-2,0-4,home,away,0.0122,0.0329,0.9549,False,False,0.9549
1681,2025-11-26,Oman,Somalia,0-0,4-0,draw,home,0.9482,0.0376,0.0142,False,False,0.9482
118,2024-03-21,South Africa,Andorra,1-1,3-0,draw,home,0.9468,0.0421,0.0111,False,False,0.9468
1772,2026-03-26,Tanzania,Liechtenstein,0-1,4-0,away,home,0.9304,0.0501,0.0195,False,False,0.9304
1799,2026-03-27,New Zealand,Finland,0-2,3-0,away,home,0.9235,0.0558,0.0207,False,False,0.9235
1296,2025-08-16,Turks and Caicos Islands,Marshall Islands,3-2,0-3,home,away,0.0235,0.0584,0.9181,False,False,0.9181
235,2024-06-06,Afghanistan,Qatar,0-0,0-3,draw,away,0.0221,0.0619,0.9160,False,False,0.9160


The Poisson model never makes draw its top pick: P(draw) sits around 20% while one side is usually above that. So every predicted result is home or away, and every actual draw is counted as a miss.

That is expected for this decision rule. The 1X2 accuracy is still the right number for "did we pick the match result", and the Brier score captures that the draw probability itself can still be well calibrated even when it is never the mode.

# Conclusion

The overall accuracy of the model is as follows:

Monte Carlo 1X2              60.740%
Monte Carlo exact score      12.724%

Correct outcomes: 1117 / 1839
Correct scores:   234 / 1839

The model never picks draws becuase it's never the most likely outcome. Thus, all 429 draws get misclassified. 